# Phase 3 - Generate ViSoLex Model A candidates on Kaggle T4
Strict pipeline: atomic chunks of 1,000, normalized sequence confidence, resume, and ordered merge. No Gemini API key is used here.

In [ ]:
!if [ -d /kaggle/working/VisolexNorm/.git ]; then cd /kaggle/working/VisolexNorm && git pull --ff-only; else git clone --depth 1 https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git /kaggle/working/VisolexNorm; fi
!pip install -q -r /kaggle/working/VisolexNorm/requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU (T4) before running.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
CODE_DIR = '/kaggle/working/VisolexNorm'
DATA_DIR = '/kaggle/input/visolexnorm-processed'  # change if needed
MODEL_A_DIR = '/kaggle/input/model-a-artifacts/model_a'  # change if needed
OUTPUT = '/kaggle/working/data/intermediate/visolex_model_a_candidates.jsonl'
CHUNK_DIR = '/kaggle/working/data/intermediate/candidate_chunks'
SMOKE_TEST = True  # run 100 samples first; set False only after smoke validation
limit_arg = '--limit 100' if SMOKE_TEST else ''
!python {CODE_DIR}/scripts/generate_candidates.py --input {DATA_DIR}/visolex_unlabeled.jsonl --checkpoint {MODEL_A_DIR} --output {OUTPUT} --chunk-dir {CHUNK_DIR} --config {CODE_DIR}/configs/candidate_generation_config.json --resume {limit_arg}

In [ ]:
import json, math, pathlib
rows = [json.loads(line) for line in pathlib.Path(OUTPUT).read_text(encoding='utf-8').splitlines() if line]
expected = 100 if SMOKE_TEST else 68411
assert len(rows) == expected and len({row['id'] for row in rows}) == expected
assert all(math.isfinite(row['model_a_confidence']) and row['sequence_token_count'] > 0 for row in rows)
print(f'Validated {len(rows)} candidates. Rerun generation to verify chunk resume.')
!cd /kaggle/working && zip -r visolex_model_a_candidates.zip data/intermediate/visolex_model_a_candidates.jsonl data/intermediate/candidate_chunks